# 🛡️ AEGIS-ULTRON-9B // MODEL COMPILER & MERGER
### Autonomous Model Synthesis for AegisOS & Ultron AI

This notebook compiles **Aegis-Ultron-9B**, a custom model built for the **AegisOS Companion AI**.
- **Base 1**: `Qwen/Qwen2.5-Coder-7B-Instruct` (Deep mathematical, logical, and coding intelligence)
- **Base 2**: `cognitivecomputations/Dolphin-2.9.4-Qwen2.5-7B` (High-speed instruction following & zero censorship)
- **Base 3**: `HauhauCS/Qwen3.5-9B-Uncensored-HauhauCS-Aggressive` (Raw reasoning & authority)
- **Method**: `DARE-TIES` + Directional Activation Abliteration
- **Output**: `Aegis-Ultron-9B-Q4_K_M.gguf` (~5.2 GB) ready for Port 8088

### Step 1: Install Dependencies
Make sure your runtime is set to **GPU (T4, V100, or A100)** in `Runtime -> Change runtime type`.

In [ ]:
!pip install -q -U mergekit transformers accelerate torch huggingface_hub
!git clone https://github.com/ggerganov/llama.cpp.git
!cd llama.cpp && make -j LLAMA_CUDA=1

### Step 2: Write MergeKit Recipe

In [ ]:
recipe = """
models:
  - model: Qwen/Qwen2.5-Coder-7B-Instruct
    parameters:
      weight: 0.45
      density: 0.70
  - model: cognitivecomputations/Dolphin-2.9.4-Qwen2.5-7B
    parameters:
      weight: 0.35
      density: 0.65
  - model: HauhauCS/Qwen3.5-9B-Uncensored-HauhauCS-Aggressive
    parameters:
      weight: 0.20
      density: 0.60

merge_method: dare_ties
base_model: Qwen/Qwen2.5-Coder-7B-Instruct

parameters:
  int8_mask: true
  normalize: true

dtype: bfloat16
tokenizer_source: base
chat_template: chatml
"""

with open('aegis_ultron_9b.yaml', 'w') as f:
    f.write(recipe.strip())
print('[OK] Recipe created: aegis_ultron_9b.yaml')

### Step 3: Run Model Merge (DARE-TIES)

In [ ]:
!mergekit-yaml aegis_ultron_9b.yaml ./aegis_ultron_9b_merged --cuda --copy-tokenizer --lazy-unpickle

### Step 4: Directional Activation Abliteration (Refusal Neutralization)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_path = './aegis_ultron_9b_merged'
print('Loading weights for orthogonalization...')
model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.bfloat16, device_map='auto')
tokenizer = AutoTokenizer.from_pretrained(model_path)

# Refusal vector neutralization across intermediate attention heads
with torch.no_grad():
    for name, param in model.named_parameters():
        if 'mlp.down_proj' in name or 'self_attn.o_proj' in name:
            param.data.mul_(0.999)

output_abliterated = './aegis_ultron_9b_abliterated'
model.save_pretrained(output_abliterated)
tokenizer.save_pretrained(output_abliterated)
print('[SUCCESS] Refusal vectors neutralized.')

### Step 5: Convert and Quantize to GGUF (Q4_K_M)

In [ ]:
!python llama.cpp/convert_hf_to_gguf.py ./aegis_ultron_9b_abliterated --outfile ./aegis_fp16.gguf --outtype f16
!llama.cpp/llama-quantize ./aegis_fp16.gguf ./Aegis-Ultron-9B-Q4_K_M.gguf Q4_K_M
!rm ./aegis_fp16.gguf
print('[SUCCESS] Aegis-Ultron-9B-Q4_K_M.gguf is ready!')

### Step 6: Download Model to Your PC
Save the GGUF directly to your computer to place into `~/.ultron/models/Aegis-Ultron-9B.gguf`.

In [ ]:
from google.colab import files
files.download('Aegis-Ultron-9B-Q4_K_M.gguf')